**Build a Standard MLP**
- **Task:** Create a class inheriting from torch.nn.Module. Build a Multilayer Perceptron (MLP) with one input layer, two hidden layers, and an output layer. Write the forward method to pass data through it. Pass a dummy tensor through to ensure the output shape matches your expectations.
- **Why:** Familiarizes you with the standard API for building network architectures.

In [1]:
import torch
import torch.nn as nn

class StandardMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim1: int, hidden_dim2: int, output_dim: int):
        super(StandardMLP, self).__init__()
        
        # Define the layers
        self.layer1 = nn.Linear(input_dim, hidden_dim1)
        self.layer2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.output_layer = nn.Linear(hidden_dim2, output_dim)
        
        # Activation function
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pass input through the network with ReLU activations
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.output_layer(x)
        return x

# --- Verification ---
if __name__ == "__main__":
    # Define dimensions
    batch_size = 8
    in_features = 10
    h1 = 32
    h2 = 16
    out_features = 3

    # Initialize the model
    model = StandardMLP(input_dim=in_features, hidden_dim1=h1, hidden_dim2=h2, output_dim=out_features)

    # Create dummy tensor (batch_size, input_dim)
    dummy_input = torch.randn(batch_size, in_features)

    # Forward pass
    output = model(dummy_input)

    print(f"Input shape:  {dummy_input.shape}")
    print(f"Output shape: {output.shape}")
    
    # Check if output shape matches expectations (batch_size, output_dim)
    assert output.shape == (batch_size, out_features), "Output shape mismatch!"

Input shape:  torch.Size([8, 10])
Output shape: torch.Size([8, 3])


**Custom Layer Implementation**
- **Task:** Do not use nn.Linear. Instead, build a custom "Linear Layer" class by explicitly defining the Weight and Bias parameters using torch.nn.Parameter. Implement "Xavier/Glorot Initialization" manually for your weights. Add this custom layer into a larger network.
- **Why:** Shows you that "layers" are just wrappers around raw tensors with registered parameters that the optimizer knows to update.

In [2]:
import math
import torch
import torch.nn as nn

class CustomLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super(CustomLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features

        # Define raw tensors as learnable parameters
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.empty(out_features))

        # Manual Xavier/Glorot Initialization
        self.reset_parameters()

    def reset_parameters(self):
        # Xavier Normal: std = sqrt(2 / (in_features + out_features))
        std = math.sqrt(2.0 / (self.in_features + self.out_features))
        nn.init.normal_(self.weight, mean=0.0, std=std)
        
        # Initialize bias to zeros
        nn.init.zeros_(self.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Linear transformation: y = x W^T + b
        return torch.matmul(x, self.weight.t()) + self.bias


class CustomMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super(CustomMLP, self).__init__()
        # Use the custom layer instead of nn.Linear
        self.layer1 = CustomLinear(input_dim, hidden_dim)
        self.layer2 = CustomLinear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu(self.layer1(x))
        x = self.layer2(x)
        return x


# --- Verification ---
if __name__ == "__main__":
    batch_size = 4
    in_features = 8
    hidden_features = 16
    out_features = 2

    # Initialize network
    model = CustomMLP(input_dim=in_features, hidden_dim=hidden_features, output_dim=out_features)

    # Create dummy tensor
    dummy_input = torch.randn(batch_size, in_features)

    # Forward pass
    output = model(dummy_input)

    print(f"Input shape:        {dummy_input.shape}")
    print(f"Output shape:       {output.shape}")
    print(f"Weight parameter:   {model.layer1.weight.shape}")
    print(f"Bias parameter:     {model.layer1.bias.shape}")
    
    # Confirm parameters are tracked by the optimizer/module
    print(f"Tracked parameters: {len(list(model.parameters()))} tensor groups")

Input shape:        torch.Size([4, 8])
Output shape:       torch.Size([4, 2])
Weight parameter:   torch.Size([16, 8])
Bias parameter:     torch.Size([16])
Tracked parameters: 4 tensor groups
